<a href="https://colab.research.google.com/github/nir-nutman/RapidFire-AI-LLM-Experimentation/blob/v2-corrected-metrics/notebooks/UCSB_catalog_rag_notebook_v3_reranked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCSB Course Catalog RAG System — v3 (corrected metrics + reranker actually applied)
## RapidFire AI Winter Competition - RAG Track

**This is NOT the competition submission.** The original notebook is unchanged at
`notebooks/UCSB_catalog_rag_notebook.ipynb` (git tag `pre-correction`); v2
(`UCSB_catalog_rag_notebook_v2.ipynb`) fixed the metrics code only. **v3 keeps all of v2's metric
fixes and additionally makes the cross-encoder reranker actually run.**

**Why v3 exists.** In `rapidfireai==0.12.9` the reranker is only created in
`LangChainRagSpec.build_pipeline()`, but the query actors rebuild their `rag_spec` around a
prebuilt retriever and never call it, so `rag.get_context()` returned the raw k=8 similarity
results. Evidence: the v2 run retrieved 6–8 unique `corpus_id`s per query even at `top_n=2`, and
`top_n=2` and `top_n=5` gave identical lists. (Read from source and the v2 output; not confirmed in a
live session.) So the original and v2 runs were effectively single-stage retrieval.

**How v3 fixes it.**
- The reranker is removed from the RAG spec (so the 4 configs share 2 contexts by chunk size, as
  intended).
- `preprocess_fn` applies `CrossEncoderReranker` itself, on the raw k=8 results, with the
  `top_n` for that config. `top_n` is swept with `List([preprocess_top2, preprocess_top5])`,
  because RapidFire's grid search expands any `List` (checked in `automl/grid_search.py`).
- Each run records `applied_top_n` and the raw (pre-rerank) retrieved IDs, so it's checkable.

**Not yet verified (will be known only after running):** that closures-free top-level functions
ship to the Ray actors correctly, that the extra output keys (`applied_top_n`,
`raw_retrieved_documents`) pass through aggregation, and that the grid yields 4 distinct runs.
The results cell checks the last two and says so loudly if not.

**Dataset:** UCSB 2009-2010 Course Catalog
**Research Question:** Which chunking and reranking strategies optimize retrieval for course catalog queries?
**Configurations Tested:** 4 (chunk_size: [128, 256] × reranking top_n: [2, 5])

## Experiment Hypothesis

Larger chunks (256 tokens) will better preserve course description context compared to smaller chunks (128 tokens). Keeping more reranked documents (top_n=5) should raise Recall and may lower Precision compared with top_n=2 — now testable because reranking is applied.

In [ ]:
# Step 1: install the package itself. NOTE: this does NOT install vllm, faiss-gpu, ray, langchain,
# etc. -- on PyPI those are optional extras, not default dependencies of rapidfireai==0.12.9.
%pip install -q rapidfireai==0.12.9

# Step 2: `rapidfireai init --evals` is what installs the evals stack. On Colab it runs
# `uv pip install -r setup/evals/requirements-colab.txt` (read from the 0.12.9 wheel, see
# rapidfireai/cli.py install_packages), which includes vllm==0.11.0, faiss-gpu-cu12,
# flashinfer-*, ray>=2.51.0, langchain*, and transformers>=4.56.1. The previous version of this
# cell checked for vllm BEFORE this line and raised, so init never ran and vllm was never installed.
!rapidfireai init --evals

# Step 3: re-pin transformers AFTER init (init's requirements file has no upper bound, so it can
# pull transformers v5). vLLM here still reads Qwen2Tokenizer.all_special_tokens_extended, which
# transformers v5 removed -> "AttributeError: Qwen2Tokenizer has no attribute
# all_special_tokens_extended" when the generator loads.
%pip install -q "transformers>=4.56.1,<5"

# Step 4: pin flashinfer-python and flashinfer-cubin to the SAME version. init's requirements list
# both as `>=0.5.3` with no upper bound; on PyPI flashinfer-python's latest is 0.7.1 but
# flashinfer-cubin's latest is 0.6.13 (no 0.7.1 exists), so pip resolved a mismatched pair and
# vLLM's worker died at startup with:
#   RuntimeError: flashinfer-cubin version (0.6.13) does not match flashinfer version (0.7.1)
# 0.6.13 is the newest version that exists for BOTH packages (checked against PyPI).
%pip install -q "flashinfer-python==0.6.13" "flashinfer-cubin==0.6.13"

# Step 5: verify, loudly. rapidfireai silently swallows a failed `import vllm`
# (automl/model_config.py: `except ImportError: _VLLM_AVAILABLE = False`), which otherwise only
# shows up cells later as "TypeError: 'NoneType' object is not callable" on RFvLLMModelConfig(...).
import importlib.metadata as md
installed_rf_version = md.version("rapidfireai")
assert installed_rf_version == "0.12.9", f"Expected rapidfireai==0.12.9, found {installed_rf_version}"

transformers_version = md.version("transformers")
assert int(transformers_version.split(".")[0]) < 5, f"Expected transformers<5, found {transformers_version}"

fi_py, fi_cubin = md.version("flashinfer-python"), md.version("flashinfer-cubin")
assert fi_py == fi_cubin, f"flashinfer-python {fi_py} != flashinfer-cubin {fi_cubin}; vLLM workers will fail to start"

try:
    import vllm
    print(f"✅ rapidfireai {installed_rf_version}, transformers {transformers_version}, "
          f"vllm {md.version('vllm')}, flashinfer {fi_py}")
except Exception:
    import traceback
    print("❌ `import vllm` failed. Real underlying error:")
    traceback.print_exc()
    print("If the error mentions torch/CUDA/binary incompatibility, use Runtime -> Restart session")
    print("(NOT 'Disconnect and delete runtime') and re-run this cell once; init replaces torch.")
    raise

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

# Ray 2.59+ turns on token authentication by default for local clusters, which Experiment(...)
# spins up under the hood. That's meant to protect shared/multi-tenant clusters; it doesn't apply
# to a single-user, ephemeral Colab session. Must be set before Ray initializes (i.e. before
# Experiment(...) is constructed below) — setting it after Ray has already started won't help.
os.environ['RAY_AUTH_MODE'] = 'disabled'

from rapidfireai import Experiment
from rapidfireai.automl import List, RFLangChainRagSpec, RFvLLMModelConfig, RFPromptManager, RFGridSearch
import re, json
from typing import List as listtype, Dict, Any

# rapidfireai.automl silently sets these to None if vllm (or its evals modules) failed to import
# at the moment automl was first imported -- and Python caches that, so installing vllm later in
# the same session does NOT fix it. Fail here, loudly, instead of as a NoneType call two cells on.
for _name, _obj in [("RFvLLMModelConfig", RFvLLMModelConfig),
                    ("RFLangChainRagSpec", RFLangChainRagSpec),
                    ("RFPromptManager", RFPromptManager)]:
    assert _obj is not None, (
        f"{_name} is None: rapidfireai.automl was imported before vllm was installed. "
        f"Runtime -> Restart session, then run the notebook from the top."
    )

## 📊 Dataset Preparation

The UCSB 2009-2010 catalog corpus, queries, and relevance judgments ship in this repository's
`data/` folder — **not** under a separate tutorial_notebooks path. (The original notebook
referenced `/content/tutorial_notebooks/rag-contexteng/datasets/ucsb_catalog/...`, a path that
only exists if RapidFire AI's own tutorial assets happen to already be present in the Colab
runtime; it was never reproducible from a clean Colab session or from this repo alone. v2 clones
this repo directly instead, so the notebook is self-contained.)

- **corpus.jsonl**: 1,304 source-section records extracted from the 490-page PDF. **Each record is
  a pre-chunking document** (mean length ~3,499 characters) — the pipeline's own text_splitter
  (chunk_size 128/256 tokens) subdivides most of these further at index time. "1,304" is the count
  of source sections, not the count of retrieval-time chunks.
- **queries.jsonl**: 30 typical student questions about courses and requirements
- **qrels.tsv**: 150 relevance judgments (5 per query). Per the original README/notebook, these
  were "generated via embedding similarity" — no data-preparation script for this exists anywhere
  in this repository's git history, so the exact method, embedding model, and whether any human
  review occurred could not be verified by audit. Treat the ground truth itself as unverified.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nir-nutman/RapidFire-AI-LLM-Experimentation.git"
REPO_DIR = Path("/content/RapidFire-AI-LLM-Experimentation")

if not REPO_DIR.exists():
    # Clone the v2-corrected-metrics branch specifically, so the cloned data/ matches this notebook.
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "v2-corrected-metrics", REPO_URL, str(REPO_DIR)],
        check=True,
    )

dataset_dir = REPO_DIR / "data"
for fname in ("corpus.jsonl", "queries.jsonl", "qrels.tsv"):
    assert (dataset_dir / fname).exists(), f"{fname} not found under {dataset_dir} — clone may have failed."
print(f"✅ Repo data staged at {dataset_dir}")

# Where v2 outputs get written back into the repo checkout, so they can be copied/committed afterward.
results_dir = REPO_DIR / "results"
results_dir.mkdir(exist_ok=True)

In [ ]:
from datasets import load_dataset
import pandas as pd
import random

# Load UCSB catalog queries
ucsb_dataset = load_dataset(
    "json",
    data_files=str(dataset_dir / "queries.jsonl"),
    split="train"
)

# Load relevance labels
qrels = pd.read_csv(str(dataset_dir / "qrels.tsv"), sep="\t")

# Sample 50% of queries for this experiment (unchanged from original, for comparability)
sample_fraction = 0.5
rseed = 1
random.seed(rseed)

sample_size = int(len(ucsb_dataset) * sample_fraction)
ucsb_dataset = ucsb_dataset.shuffle(seed=rseed).select(range(sample_size))

query_ids = set([int(qid) for qid in ucsb_dataset["query_id"]])
qrels = qrels[qrels["query_id"].isin(query_ids)]

print(f"Using {len(ucsb_dataset)} queries ({sample_fraction*100}% of dataset)")
print(f"Filtered qrels to {len(qrels)} relevance judgments")

## 🔧 RAG Pipeline Configuration

Testing 4 configurations (chunk size from the RAG spec; top_n applied in `preprocess_fn`, see below):

| Config | Chunk Size | Rerank Top-N | Hypothesis |
|--------|-----------|--------------|------------|
| 1 | 256 | 2 | High precision, focused context |
| 2 | 256 | 5 | Balanced precision & recall |
| 3 | 128 | 2 | Granular chunks, focused results |
| 4 | 128 | 5 | Granular chunks, broad coverage |

**Fixed Parameters:**
- Embedding: sentence-transformers/all-MiniLM-L6-v2
- Initial retrieval: k=8 (similarity search)
- Reranker: cross-encoder/ms-marco-MiniLM-L6-v2 (applied in `preprocess_fn`, top_n 2 or 5)
- Generator: Qwen/Qwen2.5-0.5B-Instruct

Confirmed by reading the installed `rapidfireai==0.12.9` source
(`rapidfireai/evals/rag/rag_pipeline.py`): with `vector_store=None` and
`enable_gpu_search=True`, `LangChainRagSpec.build_pipeline()` builds a **FAISS** index using
`faiss.IndexFlatL2` moved to GPU (exact L2 distance search, not approximate). With
`enable_gpu_search=False` it would instead build `faiss.IndexHNSWFlat` (approximate, CPU).

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

batch_size = 50

rag_gpu = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path=str(dataset_dir),
        glob="corpus.jsonl",
        loader_cls=JSONLoader,
        loader_kwargs={
            "jq_schema": ".",
            "content_key": "text",
            "metadata_func": lambda record, metadata: {
                "corpus_id": int(record.get("_id"))
            },
            "json_lines": True,
            "text_content": False,
        },
        sample_seed=42,
    ),
    # EXPERIMENT VARIABLE 1: Chunk size
    text_splitter=List([
        RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2", chunk_size=256, chunk_overlap=32
        ),
        RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2", chunk_size=128, chunk_overlap=32
        ),
    ]),
    embedding_cls=HuggingFaceEmbeddings,
    embedding_kwargs={
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "model_kwargs": {"device": "cuda:0"},
        "encode_kwargs": {"normalize_embeddings": True, "batch_size": batch_size},
    },
    vector_store=None,
    search_type="similarity",
    search_kwargs={"k": 8},
    # v3: NO reranker in the RAG spec. In rapidfireai==0.12.9 a reranker configured here is never
    # built in the query actors (so it silently does nothing), and top_n is not part of the context
    # hash. The reranker is applied explicitly in preprocess_top2 / preprocess_top5 instead.
    enable_gpu_search=True,
)

In [ ]:
def _get_reranker(rag, top_n):
    """Build (once per actor, cached on the rag spec) a cross-encoder reranker with this top_n."""
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder
    cache = rag.__dict__.setdefault("_rerankers_v3", {})
    if top_n not in cache:
        cache[top_n] = CrossEncoderReranker(
            model=HuggingFaceCrossEncoder(
                model_name="cross-encoder/ms-marco-MiniLM-L6-v2",
                model_kwargs={"device": "cpu"},
            ),
            top_n=top_n,
        )
    return cache[top_n]


def _preprocess(batch, rag, prompt_manager, top_n):
    """Retrieve k=8 candidates, rerank with the cross-encoder to top_n, and format prompts."""

    INSTRUCTIONS = """You are a helpful academic advisor for UCSB students.
Use the provided course catalog information to answer questions about courses,
requirements, policies, and academic programs. Be specific and cite relevant
catalog sections when possible."""

    queries = batch["query"]
    # use_reranker=False: get the raw similarity results (rag.reranker is None in the query actor
    # anyway); we rerank explicitly below so top_n is under our control.
    raw_context = rag.get_context(batch_queries=queries, use_reranker=False, serialize=False)
    reranker = _get_reranker(rag, top_n)
    all_context = [reranker.compress_documents(docs, q) for q, docs in zip(queries, raw_context)]

    raw_retrieved = [[doc.metadata["corpus_id"] for doc in docs] for docs in raw_context]
    retrieved_documents = [[doc.metadata["corpus_id"] for doc in docs] for docs in all_context]

    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"""Here is relevant information from the UCSB course catalog:

{context}

Based on this catalog information, please answer the following question:
{question}""",
                },
            ]
            for question, context in zip(queries, serialized_context)
        ],
        "retrieved_documents": retrieved_documents,       # AFTER reranking, in rank order
        "raw_retrieved_documents": raw_retrieved,         # BEFORE reranking (k=8), for auditing
        "applied_top_n": [top_n] * len(queries),
        **batch,
    }


# Plain top-level functions (not closures) so they ship cleanly to the Ray actors.
def preprocess_top2(batch, rag, prompt_manager):
    return _preprocess(batch, rag, prompt_manager, 2)


def preprocess_top5(batch, rag, prompt_manager):
    return _preprocess(batch, rag, prompt_manager, 5)

In [ ]:
def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Attach ground truth documents for evaluation"""
    batch["ground_truth_documents"] = [
        qrels[qrels["query_id"] == query_id]["corpus_id"].tolist()
        for query_id in batch["query_id"]
    ]
    return batch

## 🔧 Metrics fixes (v2)

**What Precision, Recall, and F1 are computed over:** unique source sections (`corpus_id`),
after de-duplication of the retrieved list — not the raw per-chunk retrieval list. This project's
RAG config does not expose a finer chunk-level ID: `metadata_func` (in the RAG pipeline config
cell above) only attaches `corpus_id` from the parent source record, and the text splitter isn't
configured with `add_start_index`, so no per-sub-chunk identifier survives into the Document
metadata LangChain's `get_context` returns. **Chunk-level Precision is not reported here because
the framework/configuration as currently set up doesn't expose chunk-level IDs** — adding one
would mean changing the RAG pipeline configuration itself, which is out of scope for this
metrics-only correction pass.

**What changed below, and why** (see README's "Post-competition corrections" section for the
full audit writeup):
- `compute_ndcg_at_k`: ideal relevance changed from `3` to `1`, matching the `1` used for actual
  hits. The original mismatch capped the maximum achievable NDCG@5 at 1/3 regardless of how good
  retrieval was.
- `sample_compute_metrics_fn`: `retrieved_documents` is de-duplicated with
  `list(dict.fromkeys(pred))` (preserves first-seen order) instead of `set(pred)` (which
  discards order entirely). The de-duplicated **ordered** list is passed to `compute_ndcg_at_k`
  and `compute_rr`, so rank position is preserved for the metrics that depend on it. Precision/
  Recall still use the de-duplicated set (as before), now with that explicitly documented rather
  than incidental.

In [ ]:
import math

def compute_ndcg_at_k(retrieved_docs: list, expected_docs: set, k=5):
    """Compute NDCG@k metric.

    v2 fix: ideal relevance is now 1 (matching actual-hit relevance of 1), not 3. The original
    3-vs-1 mismatch meant IDCG = 3x the best possible DCG, capping max achievable NDCG@5 at 1/3.

    v2 fix: retrieved_docs is now an ORDERED list (caller passes a de-duplicated, rank-preserving
    list, not a set) — this function was always rank-sensitive (it slices [:k] and weights by
    position), it just wasn't being given real order before.
    """
    relevance = [1 if doc in expected_docs else 0 for doc in list(retrieved_docs)[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [1] * ideal_length + [0] * (k - ideal_length)  # v2: was [3] * ideal_length
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0.0

def compute_rr(retrieved_docs: list, expected_docs: set):
    """Compute Reciprocal Rank for a single query.

    v2: retrieved_docs is now an ordered list (see sample_compute_metrics_fn) instead of a set,
    so "first relevant doc" is actually the first one retrieved, not an arbitrary set-iteration
    order.
    """
    rr = 0
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            rr = 1 / (i + 1)
            break
    return rr

def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Compute retrieval metrics per batch"""
    precisions, recalls, f1_scores, ndcgs, rrs = [], [], [], [], []
    total_queries = len(batch["query"])

    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        # v2 fix: de-duplicate corpus_ids while preserving retrieval rank order, instead of
        # set(pred) (which discards order entirely — NDCG/MRR need order, and several retrieved
        # chunks can legitimately share one corpus_id, per the dataset audit in the README).
        retrieved_ordered = list(dict.fromkeys(pred))
        retrieved_set = set(retrieved_ordered)

        true_positives = len(expected_set.intersection(retrieved_set))
        precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        # v2 fix: pass the ORDERED de-duplicated list, not the unordered set.
        ndcgs.append(compute_ndcg_at_k(retrieved_ordered, expected_set, k=5))
        rrs.append(compute_rr(retrieved_ordered, expected_set))

    return {
        "Total": {"value": total_queries},
        "Precision": {"value": sum(precisions) / total_queries},
        "Recall": {"value": sum(recalls) / total_queries},
        "F1 Score": {"value": sum(f1_scores) / total_queries},
        "NDCG@5": {"value": sum(ndcgs) / total_queries},
        "MRR": {"value": sum(rrs) / total_queries},
    }

def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Accumulate metrics across all batches (unchanged from original)"""
    num_queries_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = ["Precision", "Recall", "F1 Score", "NDCG@5", "MRR"]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                ) / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }

In [ ]:
vllm_config1 = RFvLLMModelConfig(
    model_config={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "dtype": "half",
        "gpu_memory_utilization": 0.25,
        "tensor_parallel_size": 1,
        "distributed_executor_backend": "mp",
        "enable_chunked_prefill": False,
        "enable_prefix_caching": False,
        "max_model_len": 3000,
        "disable_log_stats": True,
        "enforce_eager": True,
        "disable_custom_all_reduce": True,
    },
    sampling_params={
        "temperature": 0.8,
        "top_p": 0.95,
        "max_tokens": 128,
    },
    rag=rag_gpu,
    prompt_manager=None,
)

batch_size = 3

config_set = {
    "vllm_config": vllm_config1,
    "batch_size": batch_size,
    "preprocess_fn": List([preprocess_top2, preprocess_top5]),  # EXPERIMENT VARIABLE 2: rerank top_n
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name": "normal",
        "confidence_level": 0.95,
        "use_fpc": True,
    },
}

In [ ]:
config_group = RFGridSearch(config_set)

## 🚀 Running Experiments (v3)

This cell executes all 4 configurations in parallel using RapidFire AI's experiment orchestration
system, using the corrected metrics code above. Uses a separate `experiment_name` from the
original run so it doesn't collide with any existing RapidFire experiment state for
`ucsb-catalog-rag-exp1`.

**Expected Runtime:** ~20-30 minutes on Colab T4 GPU (unchanged — only metrics scoring changed,
not retrieval/generation cost).

In [ ]:
experiment = Experiment(experiment_name="ucsb-catalog-rag-exp1-v3", mode="evals")

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8855)

In [ ]:
import sys, traceback

# Tee all output to Google Drive (persistent), not just the Colab cell display, so the real
# root cause survives even if a crash severe enough to reset the runtime happens mid-run.
log_path = "/content/drive/MyDrive/run_evals_debug_log_v3.txt"
with open(log_path, "w") as logf:
    class Tee:
        def __init__(self, *streams): self.streams = streams
        def write(self, data):
            for s in self.streams: s.write(data)
        def flush(self):
            for s in self.streams: s.flush()

    orig_stdout, orig_stderr = sys.stdout, sys.stderr
    sys.stdout = Tee(orig_stdout, logf)
    sys.stderr = Tee(orig_stderr, logf)
    try:
        results = experiment.run_evals(
            config_group=config_group,
            dataset=ucsb_dataset,
            num_actors=1,
            num_shards=4,
            seed=42,
        )
    except Exception:
        traceback.print_exc()
        raise  # do NOT swallow: otherwise this cell looks finished and the failure only surfaces
               # later as "NameError: name 'results' is not defined"
    finally:
        sys.stdout, sys.stderr = orig_stdout, orig_stderr

print(f"✅ run_evals finished; full output saved to {log_path}")

## 📊 Results Analysis (v2)

The table below shows the corrected final metrics for all 4 configurations.

**Key Metrics (same definitions as original, Precision/Recall/F1 now explicitly over
de-duplicated unique `corpus_id`, NDCG@5/MRR now rank-order-aware):**
- **Precision**: Of unique retrieved source sections, what % were relevant?
- **Recall**: Of all relevant source sections, what % were retrieved?
- **F1 Score**: Harmonic mean of Precision and Recall
- **NDCG@5**: How well were relevant documents ranked? (0-1 scale, 1 = perfect ranking — v2 fix
  removes the old 1/3 ceiling)
- **MRR**: How quickly was the first relevant document found, in true retrieval rank order (v2 fix)

In [ ]:
if "results" not in globals():
    raise RuntimeError(
        "`results` is not defined: the run_evals cell did not complete. Scroll up to that cell's "
        "error, or read /content/drive/MyDrive/run_evals_debug_log_v3.txt. Don't debug this cell."
    )

# --- v3 sanity checks first: did the grid give 4 distinct runs, and did reranking really apply? ---
applied_top_n = {}
for run_id, (agg, _m) in results.items():
    print(f"run {run_id}: aggregated_results keys = {list(agg.keys())}")
    if "applied_top_n" not in agg or "retrieved_documents" not in agg:
        raise RuntimeError(
            f"run {run_id}: 'applied_top_n'/'retrieved_documents' missing from aggregated_results, so the "
            f"extra keys did not pass through aggregation. Share the printed keys above."
        )
    vals = sorted(set(agg["applied_top_n"]))
    assert len(vals) == 1, f"run {run_id}: mixed applied_top_n values {vals}"
    applied_top_n[run_id] = vals[0]
    n_ret = [len(set(p)) for p in agg["retrieved_documents"]]
    n_raw = [len(set(p)) for p in agg["raw_retrieved_documents"]]
    print(f"run {run_id}: applied top_n={vals[0]}; unique ids/query AFTER rerank {min(n_ret)}-{max(n_ret)}, "
          f"BEFORE rerank {min(n_raw)}-{max(n_raw)}")
    assert max(n_ret) <= vals[0], (
        f"run {run_id}: top_n={vals[0]} but a query kept {max(n_ret)} unique ids; reranking did NOT cut as expected."
    )
assert len(results) == 4, f"expected 4 runs, got {len(results)}"
assert sorted(applied_top_n.values()) == [2, 2, 5, 5], f"expected two runs each of top_n 2 and 5, got {applied_top_n}"

results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v
     for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])
# The framework's own top_n column (if any) can't be trusted here; use what preprocess actually applied.
results_df["top_n"] = results_df["run_id"].map(applied_top_n)

display_cols = ['run_id', 'chunk_size', 'top_n', 'Precision', 'Recall', 'F1 Score', 'NDCG@5', 'MRR']
display(results_df[display_cols])

csv_path = results_dir / "ucsb_catalog_results_v3.csv"
results_df.to_csv(csv_path, index=False)
print(f"✅ v3 results saved to {csv_path}")

# Per-query ordered, de-duplicated IDs, both after and before reranking, so this can be audited later.
retrieved_ids_v3 = {}
for run_id, (agg, metrics_dict) in results.items():
    chunk = metrics_dict.get("chunk_size")
    chunk = chunk.get("value") if isinstance(chunk, dict) else chunk
    retrieved_ids_v3[str(run_id)] = {
        "chunk_size": chunk,
        "top_n": applied_top_n[run_id],
        "retrieved_ids_by_query": {
            str(qid): list(dict.fromkeys(pred))
            for qid, pred in zip(agg["query_id"], agg["retrieved_documents"])
        },
        "raw_retrieved_ids_by_query": {
            str(qid): list(dict.fromkeys(pred))
            for qid, pred in zip(agg["query_id"], agg["raw_retrieved_documents"])
        },
    }

ids_path = results_dir / "retrieved_ids_v3.json"
with open(ids_path, "w") as f:
    json.dump(retrieved_ids_v3, f, indent=2)
print(f"✅ per-query retrieved IDs (after and before reranking) saved to {ids_path}")

## 🔍 Key Findings (v3) — fill in after running the cells above

**Template.** Replace the placeholders with what `results_df`, `retrieved_ids_v3.json` and the
sanity-check lines above actually show.

### Did reranking apply?
[from the results cell: unique ids/query AFTER vs BEFORE rerank for each run]

### Corrected metrics (with reranking)
| chunk | top_n | Precision | Recall | F1 | NDCG@5 | MRR |
|---|---|---|---|---|---|---|
| 256 | 2 | [ ] | [ ] | [ ] | [ ] | [ ] |
| 256 | 5 | [ ] | [ ] | [ ] | [ ] | [ ] |
| 128 | 2 | [ ] | [ ] | [ ] | [ ] | [ ] |
| 128 | 5 | [ ] | [ ] | [ ] | [ ] | [ ] |

### Observations — only what the numbers show
- Chunk size: [ ]
- top_n: [compare 2 vs 5; remember Precision is over unique corpus_id and with top_n=2 at most 2 per query]
- Reranked vs the v2 (no reranker) results: [ ]
- Caveats: 15 queries, no confidence intervals; relevance labels were auto-generated and are unverified.

### Do not carry over
"students find information 20% faster", "answers are 9% more accurate", and "real-time confidence
intervals showed superiority" were unsupported in the original; MRR/Precision are retrieval metrics.

In [ ]:
experiment.end()
print("v3 experiment completed")

In [ ]:
# Save to Google Drive for backup
results_df.to_csv('/content/drive/MyDrive/ucsb_catalog_results_v3.csv', index=False)
print("✅ Results also saved to Google Drive")
print()
print("Files now inside the repo checkout at", REPO_DIR, ":")
print(" -", csv_path)
print(" -", ids_path)
print()
print("Download them with:")
print("  from google.colab import files")
print("  files.download(str(csv_path)); files.download(str(ids_path))")